In [1]:
import pandas as pd
import json
import zipfile
import io

print("Loading datasets...")

# 1. Load the Metacritic CSV
# Note: Check if the title column in this CSV is 'title', 'Title', or 'game_title'. Update 'title' below if needed.
df_meta = pd.read_csv('dataset_metacritic_scraper_2025-02-15.csv')

# 2. Load the Steam JSON
with open('games.json', 'r', encoding='utf-8') as f:
    steam_data = json.load(f)

# Convert the JSON dictionary into a Pandas DataFrame
steam_rows = []
for app_id, info in steam_data.items():
    steam_rows.append({
        'App_ID': app_id,
        'Steam_Title': info.get('name', ''),
        'Price': info.get('price', 0),
        'Release_Date': info.get('release_date', '')
    })
df_steam = pd.DataFrame(steam_rows)

print("Data loaded successfully!")

Loading datasets...


C:\Users\PMLS\AppData\Local\Temp\ipykernel_24936\1355221320.py:10: DtypeWarning: Columns (0: metascore, 1: platformReviews/8/name, 2: platformReviews/8/url, 3: platformReviews/9/name, 4: platformReviews/9/url, 5: platformReviews/10/name, 6: platformReviews/10/url, 7: platformReviews/11/name, 8: platformReviews/11/url, 9: platforms/8, 10: platforms/9, 11: platforms/10, 12: platforms/11) have mixed types. Specify dtype option on import or set low_memory=False.
  df_meta = pd.read_csv('dataset_metacritic_scraper_2025-02-15.csv')


Data loaded successfully!


In [2]:
print("Cleaning titles and merging...")

# Convert both title columns to lowercase and remove extra spaces so they match perfectly
df_meta['match_title'] = df_meta['title'].astype(str).str.lower().str.strip()
df_steam['match_title'] = df_steam['Steam_Title'].astype(str).str.lower().str.strip()

# Perform the INNER JOIN 
df_merged = pd.merge(df_steam, df_meta, on='match_title', how='inner')

# Drop the temporary matching column to keep it clean
df_merged = df_merged.drop(columns=['match_title'])

print(f"Merge complete! Final dataset contains {len(df_merged)} overlapping games.")

Cleaning titles and merging...


C:\Users\PMLS\AppData\Local\Temp\ipykernel_24936\636010533.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_meta['match_title'] = df_meta['title'].astype(str).str.lower().str.strip()


Merge complete! Final dataset contains 1788 overlapping games.


In [3]:
print("Processing text metrics from ZIP archive...")

review_stats = []

# Open the zip file in read mode
with zipfile.ZipFile('Game Reviews.zip', 'r') as z:
    all_files_in_zip = z.namelist()
    
    # Loop only through the games that survived our inner join
    for app_id in df_merged['App_ID']:
        
        # Find the specific CSV for this game inside the zip file
        # The zip might have them in a folder, so we check if the filename ends with the App_ID
        matching_files = [f for f in all_files_in_zip if f.endswith(f"{app_id}.csv")]
        
        if matching_files:
            target_file = matching_files[0]
            
            # Read that specific file directly from the zip into memory
            with z.open(target_file) as f:
                df_reviews = pd.read_csv(f)
                
                # Basic cleaning: remove empty reviews
                if 'review' in df_reviews.columns: # Update 'review' if the column name is different
                    df_reviews = df_reviews.dropna(subset=['review'])
                    
                    # Calculate metrics
                    total_reviews = len(df_reviews)
                    avg_word_count = df_reviews['review'].apply(lambda x: len(str(x).split())).mean()
                    
                    review_stats.append({
                        'App_ID': app_id,
                        'Total_Reviews': total_reviews,
                        'Avg_Review_Word_Count': round(avg_word_count, 1)
                    })

# Convert the stats into a dataframe and merge them into our main dataset
df_stats = pd.DataFrame(review_stats)
if not df_stats.empty:
    df_merged = pd.merge(df_merged, df_stats, on='App_ID', how='left')

# Save the final deliverable!
df_merged.to_csv('Final_Cleaned_Dataset.csv', index=False)

print("Pipeline complete! 'Final_Cleaned_Dataset.csv' has been generated.")

Processing text metrics from ZIP archive...
Pipeline complete! 'Final_Cleaned_Dataset.csv' has been generated.
